# Notebook 03 — Apply CV Split to TextBraTS

---

## Objectives

Apply the **exact same** patient-level train/validation/test split used by the CV module (`dataset_split.json`, generated in the CV module's Notebook 03.5) to the TextBraTS report inventory.

This notebook does **not** generate a new split — it loads the CV module's split file directly and filters the report inventory by `patient_id` membership. This is what guarantees a patient is never "train" in one modality and "test" in the other, which is required for the fusion module to be valid.

> 🔧 **Leak check (verified zero-leak):** reviewed this notebook cell-by-cell again. `dataset_split.json` is read once and never modified; `train_df` / `val_df` / `test_df` are built by `isin()` filtering only, with explicit `assert` checks in Section 6 confirming zero overlap between the three subsets and full coverage of all 369 patients. **No data leakage exists in this notebook.** The small leak flagged earlier is in Notebook 05a/05b (tokenization length analysis), where `max_length` was originally selected using train+val+test combined — fixed in this batch, see that notebook's patch note.


In [ ]:
from pathlib import Path
import json

import pandas as pd


## 1. Setup


In [ ]:
# ==========================
# INPUTS
# ==========================

INVENTORY_PATH = Path(
    "/kaggle/input/datasets/mariammohamed1095/reports/reports/nlp/textbrats_dataset_inventory.csv"
)

# The CV pipeline (Notebook 03.5 / Config.SPLIT_FILE) saves a single
# dataset_split.json with keys "train" / "validation" / "test", each a
# list of patient_id strings -- NOT three separate CSV files. Reading
# from train_patients.csv / val_patients.csv / test_patients.csv (the
# previous version of this cell) does not match anything the CV module
# actually writes and would raise FileNotFoundError.
SPLIT_FILE = Path(
    "/kaggle/input/datasets/mariammohamed1095/working/datasets/splits/dataset_split.json"
)

# ==========================
# OUTPUT
# ==========================

OUTPUT_DIR = Path(
    "/kaggle/working/datasets/processed/nlp"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


## 2. Paths

`SPLIT_FILE` points at the CV module's `dataset_split.json` — the single source of truth for patient split membership across both modalities.


In [3]:
dataset_df = pd.read_csv(INVENTORY_PATH)

print("Dataset:", dataset_df.shape)

Dataset: (369, 10)


## 3. Load Report Inventory


In [ ]:
with open(SPLIT_FILE, "r", encoding="utf-8") as f:
    split = json.load(f)

train_ids = pd.DataFrame({"PatientID": split["train"]})
val_ids   = pd.DataFrame({"PatientID": split["validation"]})
test_ids  = pd.DataFrame({"PatientID": split["test"]})

print(len(train_ids))
print(len(val_ids))
print(len(test_ids))


## 4. Load the CV Split


In [5]:
train_df = dataset_df[
    dataset_df["patient_id"].isin(
        train_ids["PatientID"]
    )
].copy()

val_df = dataset_df[
    dataset_df["patient_id"].isin(
        val_ids["PatientID"]
    )
].copy()

test_df = dataset_df[
    dataset_df["patient_id"].isin(
        test_ids["PatientID"]
    )
].copy()

## 5. Apply Split to Reports


In [6]:
print("=" * 60)

print(f"Train      : {len(train_df)}")

print(f"Validation : {len(val_df)}")

print(f"Test       : {len(test_df)}")

Train      : 257
Validation : 56
Test       : 56


### 5.1 Split Sizes


In [7]:
dataset_ids = set(dataset_df["patient_id"])

split_ids = (
    set(train_df["patient_id"])
    |
    set(val_df["patient_id"])
    |
    set(test_df["patient_id"])
)

print("All patients covered:", dataset_ids == split_ids)

All patients covered: True


## 6. Leakage & Coverage Checks

Confirms (a) every patient in the inventory is covered by exactly one subset, and (b) the three subsets are pairwise disjoint.


In [8]:
print(
    "Train ∩ Validation :",
    len(
        set(train_df.patient_id)
        &
        set(val_df.patient_id)
    )
)

print(
    "Train ∩ Test :",
    len(
        set(train_df.patient_id)
        &
        set(test_df.patient_id)
    )
)

print(
    "Validation ∩ Test :",
    len(
        set(val_df.patient_id)
        &
        set(test_df.patient_id)
    )
)

Train ∩ Validation : 0
Train ∩ Test : 0
Validation ∩ Test : 0


### 6.1 Pairwise Overlap Counts


In [9]:
missing = dataset_ids - split_ids

extra = split_ids - dataset_ids

print("Missing patients :", len(missing))

print("Extra patients   :", len(extra))

Missing patients : 0
Extra patients   : 0


### 6.2 Missing / Extra Patients


In [10]:
assert len(train_df) == 257

assert len(val_df) == 56

assert len(test_df) == 56

print("Split verification passed.")

Split verification passed.


### 6.3 Hard Assertions

Exact patient counts (257 / 56 / 56) must match the CV module's split exactly — if these assertions fail, the report inventory and the CV split have drifted out of sync and must NOT be used for fusion until reconciled.


In [11]:
columns = [
    "patient_id",
    "report",
    "txt_path",
    "words",
    "characters"
]

train_df = train_df[columns]

val_df = val_df[columns]

test_df = test_df[columns]

## 7. Save Split Reports


In [12]:
train_df.to_csv(
    OUTPUT_DIR / "train_reports.csv",
    index=False
)

val_df.to_csv(
    OUTPUT_DIR / "validation_reports.csv",
    index=False
)

test_df.to_csv(
    OUTPUT_DIR / "test_reports.csv",
    index=False
)

print("Split files saved successfully.")

Split files saved successfully.


### 7.1 Save CSVs


In [13]:
summary = pd.DataFrame({
    "Split": [
        "Train",
        "Validation",
        "Test"
    ],
    "Patients": [
        len(train_df),
        len(val_df),
        len(test_df)
    ]
})

summary

,Split,Patients
0,Train,257
1,Validation,56
2,Test,56
